# 第 15 章　分析の統合

問い・測定・日付の対応・比較条件を一つの分析につなぎ，気象の記事を書きます。

このノートの前半は，本文の説明と一緒に実行します。各場面で先に短い問いを読み，講師の案内に合わせてセルを動かし，出力の見所を確認します。後半は約20分の実習です。前半のコードは動く見本として用意してあり，作図コードをすべて書き写す必要はありません。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。CSV は，手元にファイルがなければ公開 URL から読み込みます。ローカルでは，配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置いて実行できます。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

file_name = "jma-temperature-2024-08.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

temperatures = pd.read_csv(
    csv_path, dtype={"station_code": str}, parse_dates=["date"],
)
station_codes = ["47626", "47807"]
station_labels = ["Kumagaya", "Fukuoka"]
station_colors = ["#195b8a", "#a7421f"]

temperatures[["station", "date", "max_temp_c", "min_temp_c"]].head()

## 本文と一緒に実行

### 1. 「暑かった」を測定へ置き換える

8月7日の二地点では，日最高気温と日最低気温のどちらを比べるかで大小が入れ替わります。原表で地点・日付・項目を確認してから，三つの気温を描きます。

In [ ]:
example_date = pd.Timestamp("2024-08-07")
temperature_fields = ["mean_temp_c", "max_temp_c", "min_temp_c"]
one_date = temperatures.loc[temperatures["date"] == example_date]
one_date[["station", "date"] + temperature_fields]

In [ ]:
plt.rcParams["figure.figsize"] = (7.2, 2.7)
fig, ax = plt.subplots(layout="constrained")
for level, code, color in zip([1, 0], station_codes, station_colors):
    day = one_date.loc[one_date["station_code"] == code].iloc[0]
    low, mean, high = day["min_temp_c"], day["mean_temp_c"], day["max_temp_c"]
    ax.hlines(level, low, high, color=color, linewidth=3)
    ax.scatter([low, high], [level, level], color=color, marker="|", s=190)
    ax.scatter(mean, level, color=color, marker="D", s=41, zorder=3)
    for value, name in [(low, "Min"), (mean, "Mean"), (high, "Max")]:
        ax.annotate(f"{name} {value:.1f}", (value, level), xytext=(0, 11),
                    textcoords="offset points", ha="center", color=color, fontsize=9)
ax.set(xlabel="Temperature (°C)", yticks=[1, 0], yticklabels=station_labels,
       xlim=(22.5, 37.5), ylim=(-0.4, 1.6))
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
plt.show()

**出力の見所：**線の左右の端は日最低・日最高気温，ひし形は日平均です。日平均は両端の中点でしょうか。日最低気温は一日全体の最低で，夜間の最低気温を直接集計した列ではありません。

### 2. 分母を確かめて日数を数える

一地点31行であれば，全31日が一回ずつそろうと言えるでしょうか。地点と日付の重複，値の欠損，暦との対応を確認します。

In [ ]:
station_day = ["station_code", "date"]
by_station = temperatures.groupby(["station_code", "station"])
coverage = by_station.agg(
    rows=("date", "size"),
    dates=("date", "nunique"),
    maximum_values=("max_temp_c", "count"),
    minimum_values=("min_temp_c", "count"),
    mean_values=("mean_temp_c", "count"),
)
print("Duplicate station-date rows:", temperatures.duplicated(station_day).sum())
coverage

In [ ]:
august = pd.date_range("2024-08-01", "2024-08-31", freq="D")
for (code, station), station_rows in by_station:
    dates = pd.DatetimeIndex(station_rows["date"]).sort_values()
    print(code, station, dates.equals(august))

ordered_temperatures = (
    (temperatures["min_temp_c"] <= temperatures["mean_temp_c"])
    & (temperatures["mean_temp_c"] <= temperatures["max_temp_c"])
)
print("Rows with minimum ≤ mean ≤ maximum:", ordered_temperatures.sum())

### 記録が一日欠けたら

次は2024年8月25〜27日の日最高気温です。**練習のために福岡の26日の行を外しています。**

| 地点 | 日付 | 日最高気温（℃） |
|---|---|---:|
| 熊谷 | 2024-08-25 | 34.0 |
| 熊谷 | 2024-08-26 | 35.8 |
| 熊谷 | 2024-08-27 | 33.8 |
| 福岡 | 2024-08-25 | 32.4 |
| 福岡 | 2024-08-27 | 37.4 |

1. 一行は何を表しますか。25日と27日に限定すると，各地点で35℃以上だったのは何日ですか。分子と分母を示してください。
2. 二地点の25〜27日をすべて比べるには，元の資料のどの値を確認しますか。

（ここに記入）


In [ ]:
criteria = temperatures.assign(
    maximum_at_least_35=temperatures["max_temp_c"] >= 35,
    minimum_at_least_25=temperatures["min_temp_c"] >= 25,
)
criteria[["station", "date", "max_temp_c", "maximum_at_least_35"]].loc[
    criteria["max_temp_c"] == 35
]

In [ ]:
monthly_counts = criteria.groupby("station_code").agg(
    days=("date", "size"),
    days_max_at_least_35=("maximum_at_least_35", "sum"),
    days_min_at_least_25=("minimum_at_least_25", "sum"),
)
monthly_counts["percent_max_at_least_35"] = (
    100 * monthly_counts["days_max_at_least_35"] / monthly_counts["days"]
)
monthly_counts["percent_min_at_least_25"] = (
    100 * monthly_counts["days_min_at_least_25"] / monthly_counts["days"]
)
monthly_counts.round(1)

**出力の見所：**日最高35℃以上は熊谷20日・福岡17日，日最低25℃以上は熊谷24日・福岡30日です。35.0℃の日を含める条件はどの記号で表されていますか。

### 3. 同じ日を対応させる

月の日数だけでは見えない日ごとの差を確かめます。日付順に描いた後，日付をキーに結合します。図の一点は何を一組にした値か，説明してください。

In [ ]:
import matplotlib.dates as mdates
kuma_code, fuku_code = station_codes
rules = [("max_temp_c", 35, "Daily maximum ≥ 35 °C"),
         ("min_temp_c", 25, "Daily minimum ≥ 25 °C")]
plt.rcParams["figure.figsize"] = (7.2, 5.4)
fig, axes = plt.subplots(2, 1, sharex=True, sharey=True, layout="constrained")
for ax, (measure, threshold, title) in zip(axes, rules):
    for code, label, color, style in zip(
        station_codes, station_labels, station_colors, ["-", "--"],
    ):
        station_rows = temperatures.loc[temperatures["station_code"] == code]
        station_rows = station_rows.sort_values("date")
        ax.plot(station_rows["date"], station_rows[measure], label=label,
                color=color, linestyle=style, marker=".", markersize=3.6)
    ax.axhline(threshold, color="#6e7880", linestyle=":", linewidth=1.1)
    ax.set(ylabel="Temperature (°C)", title=title.split(" ≥ ")[0], ylim=(22, 40))
axes[0].legend(frameon=False, ncol=2, loc="lower left")
axes[-1].set_xticks(august[::5])
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
axes[-1].set_xlabel("Date (2024)")
plt.show()

In [ ]:
pair_fields = ["date", "max_temp_c", "min_temp_c"]
kumagaya = temperatures.loc[temperatures["station_code"] == kuma_code, pair_fields]
fukuoka = temperatures.loc[temperatures["station_code"] == fuku_code, pair_fields]

paired = kumagaya.merge(
    fukuoka,
    on="date",
    how="outer",
    validate="one_to_one",
    suffixes=("_kumagaya", "_fukuoka"),
    indicator=True,
).sort_values("date")
paired["_merge"].value_counts()

In [ ]:
paired["maximum_gap_c"] = (
    paired["max_temp_c_kumagaya"] - paired["max_temp_c_fukuoka"]
).round(1)
paired["minimum_gap_c"] = (
    paired["min_temp_c_kumagaya"] - paired["min_temp_c_fukuoka"]
).round(1)
paired.loc[paired["date"] == example_date, ["date", "maximum_gap_c", "minimum_gap_c"]]

In [ ]:
plt.rcParams["figure.figsize"] = (6.5, 4.8)
fig, ax = plt.subplots(layout="constrained")
ax.axvline(0, color="#77838c", linestyle=":", linewidth=1)
ax.axhline(0, color="#77838c", linestyle=":", linewidth=1)
ax.scatter(paired["maximum_gap_c"], paired["minimum_gap_c"], s=30, color="#465f72")
for day_number, offset in [(7, (-67, -16)), (12, (-28, 8)), (17, (-35, 17)), (22, (9, 3))]:
    row = paired.loc[paired["date"].dt.day == day_number].iloc[0]
    coords = (row["maximum_gap_c"], row["minimum_gap_c"])
    ax.scatter(*coords, s=47, facecolor="white", edgecolor=station_colors[0], zorder=3)
    ax.annotate(f"Aug {day_number}", coords, xytext=offset,
                textcoords="offset points", fontsize=9,
                arrowprops={"arrowstyle": "-", "color": "#697a88", "linewidth": 0.7})
ax.set(xlabel="Daily maximum: Kumagaya − Fukuoka (°C)",
       ylabel="Daily minimum: Kumagaya − Fukuoka (°C)",
       xlim=(-6, 6), ylim=(-5.7, 3), xticks=[-6, -4, -2, 0, 2, 4, 6])
plt.show()

In [ ]:
gap_counts = pd.DataFrame({
    "Kumagaya higher": (paired[["maximum_gap_c", "minimum_gap_c"]] > 0).sum(),
    "Equal": (paired[["maximum_gap_c", "minimum_gap_c"]] == 0).sum(),
    "Fukuoka higher": (paired[["maximum_gap_c", "minimum_gap_c"]] < 0).sum(),
})
gap_counts

**出力の見所：**右下の点は日最高気温が熊谷で高く，日最低気温が福岡で高い日です。日最低気温が福岡で高い28日という数は，25℃以上の30日という数と何が違うでしょうか。

### 4. 二地点の該当日は重なるか

熊谷20日と福岡17日を足すと，「少なくとも片方で35℃以上だった暦日の日数」になるでしょうか。四通りのTrue/Falseを数えます。

In [ ]:
kuma_reached = paired["max_temp_c_kumagaya"] >= 35
fuku_reached = paired["max_temp_c_fukuoka"] >= 35
pd.crosstab(
    kuma_reached.rename("Kumagaya"),
    fuku_reached.rename("Fukuoka"),
)

**出力の見所：**両地点で該当した日を二回数えていないか確認します。表の各セルから，地点別の20日と17日を作り直してください。

### 5. 基準だけを動かす

猛暑日の定義は35℃以上です。別の問いとして34℃以上・36℃以上も調べると，どちらが多いかは同じでしょうか。期間は8月全体のままにして関数へ渡します。

In [ ]:
def count_days_at_or_above(data, temperature_column, threshold):
    """Count qualifying days for each station."""
    values = data[temperature_column]
    reached = values >= threshold
    return reached.groupby(data["station_code"]).sum()


thresholds = [34, 35, 36]
counts_by_threshold = pd.DataFrame(
    [count_days_at_or_above(temperatures, "max_temp_c", limit) for limit in thresholds],
    index=pd.Index(thresholds, name="threshold_c"),
)
counts_by_threshold

In [ ]:
plt.rcParams["figure.figsize"] = (6.5, 3.7)
fig, ax = plt.subplots(layout="constrained")
for code, label, color, marker, style in zip(
    station_codes, station_labels, station_colors, ["s", "o"], ["-", "--"],
):
    ax.plot(counts_by_threshold.index, counts_by_threshold[code], marker=marker,
            linestyle=style, color=color, label=label)
ax.set(xlabel="Daily maximum temperature threshold (°C)", ylabel="Days at or above",
       xticks=thresholds, ylim=(0, 31), xlim=(33.85, 36.15))
ax.legend(frameon=False)
plt.show()

In [ ]:
monthly_maximum = by_station["max_temp_c"].agg(["mean", "median", "max"])
monthly_maximum.round(2)

**出力の見所：**34℃から35℃に基準を上げたとき，どの温度範囲の日が数える対象から外れますか。平均・中央値・最大が答える問いとも比べ，記事で最初に決めた問いに合う数を選びます。

### 6. 図から一文を書く

講義で読んだ数を一つ選び，地点・期間・測定項目・分母が分かる一文を書いてください。別の年にも同じ傾向があると述べるには，何を追加で調べるかも一つ挙げます。

（ここに一文と次に調べることを書く）

## Jupyter 実習（約20分）

二地点の気温を紹介する短い記事を作ります。**気温の項目と，比較の仕方の二つ**を，読み手が知りたいことから選びます。追加で比べる期間は全員共通の1〜15日と16〜31日です。前半と後半を同じ関数へ別々に渡し，一回の全セル実行で両方の結果を残します。

## 1. 読み手と比較を決める（約4分）

記事を読む人に何を伝えたいか，一文で書きます。例えば，一日の高い気温に着目した記事と，一日の最も低い気温にも着目した記事では，読む列が変わります。

| 選ぶもの | 指定する値 | 問いとの対応 |
|---|---|---|
| 気温の項目 | `"max_temp_c"` または `"min_temp_c"` | 日最高気温，または日最低気温を比べる。 |
| 比較の仕方 | `"threshold"` または `"same_day"` | 基準に達した日数，または同じ日に相手の地点より高かった日数を比べる。 |

`threshold` の基準は，日最高なら35℃，日最低なら25℃です。今回は基準は動かしません。`same_day` では日付で二地点を対応させ，どちらが高かったかを数えます。同じ値の日は『高かった日』に含めません。

例えば二地点とも35℃以上の日は，`threshold` なら両地点に一日ずつ入ります。`same_day` なら，日最高気温が高い方に一日が入ります。この違いを手掛かりに，記事が答える問いを決めてください。

- 読み手に伝えたいこと：
- 選んだ気温の項目と理由：
- 選んだ比較の仕方と，もう一方では答えが異なる理由：


In [ ]:
selected_measure = None  # "max_temp_c" または "min_temp_c"
comparison_kind = None  # "threshold" または "same_day"


## 2. 二つの期間を同じ規則で比べる（約8分）

次の関数は準備済みです。`threshold` では本文の `count_days_at_or_above()` を使います。`same_day` では本文と同じように日付で二地点を結合し，相手より高い日を数えます。どちらも，地点ごとの該当日数 `count`，比較に使った日数 `days`，割合 `share_pct` を返します。この月の気温に欠損がなく，二地点とも全31日がそろうことは講義の確認を使います。

関数を一から書く課題ではありません。選んだ条件に対応する処理と，返された日数の意味を読みます。


In [ ]:
def compare_temperature_period(records, measure, kind):
    if kind == "threshold":
        threshold = 35 if measure == "max_temp_c" else 25
        counts = count_days_at_or_above(records, measure, threshold)
        days = records.groupby("station_code")[measure].count()
    else:
        first = records.loc[records["station_code"] == station_codes[0], ["date", measure]]
        second = records.loc[records["station_code"] == station_codes[1], ["date", measure]]
        pairs = first.merge(second, on="date", how="inner", validate="one_to_one",
                            suffixes=("_kumagaya", "_fukuoka"))
        first_values = pairs[measure + "_kumagaya"]
        second_values = pairs[measure + "_fukuoka"]
        counts = pd.Series(
            [(first_values > second_values).sum(), (second_values > first_values).sum()],
            index=station_codes,
        )
        days = pd.Series(len(pairs), index=station_codes)
    result = pd.DataFrame({"count": counts, "days": days}).reindex(station_codes)
    result["share_pct"] = 100 * result["count"] / result["days"]
    return result


In [ ]:
valid_measure = selected_measure in ["max_temp_c", "min_temp_c"]
valid_comparison = comparison_kind in ["threshold", "same_day"]
first_result = None
if valid_measure and valid_comparison:
    first_half = temperatures.loc[temperatures["date"].between("2024-08-01", "2024-08-15")]
    first_result = compare_temperature_period(first_half, selected_measure, comparison_kind)
    display(first_result.round(1))
else:
    print("気温の項目と比較の仕方を選んでから，このセルを実行します。")


ここまでは1〜15日の結果です。まず前半について記事の一文を書きます。後半でも二地点の大小は同じだと思いますか。理由を添えて予想してから，次のセルで16〜31日の結果を確かめます。前半のコードと結果はそのまま残します。

- 前半の結果だけを使った記事の一文：
- 後半についての予想と理由：


In [ ]:
period_results = None
if valid_measure and valid_comparison:
    second_half = temperatures.loc[temperatures["date"].between("2024-08-16", "2024-08-31")]
    second_result = compare_temperature_period(second_half, selected_measure, comparison_kind)
    period_results = pd.concat(
        {"Aug 1–15": first_result, "Aug 16–31": second_result}, names=["period", "station_code"]
    )
    display(period_results.round(1))
    if comparison_kind == "same_day":
        for label, result in [("Aug 1–15", first_result), ("Aug 16–31", second_result)]:
            equal_days = result["days"].iloc[0] - result["count"].sum()
            print(label, "— equal temperature on", int(equal_days), "day(s)")
else:
    print("気温の項目と比較の仕方を選ぶと，両期間の比較表が出ます。")


In [ ]:
if period_results is None:
    print("二つの選択を記入すると，前半と後半の比較図が出ます。")
else:
    measure_title = "Daily maximum" if selected_measure == "max_temp_c" else "Daily minimum"
    if comparison_kind == "threshold":
        threshold = 35 if selected_measure == "max_temp_c" else 25
        comparison_title = f"Days at or above {threshold} °C"
    else:
        comparison_title = "Days warmer than the other station"
    fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(7.2, 3.2),
                             layout="constrained")
    for ax, label in zip(axes, ["Aug 1–15", "Aug 16–31"]):
        result = period_results.loc[label]
        bars = ax.barh(station_labels, result["share_pct"], color=station_colors)
        ax.bar_label(bars, labels=[f"{int(n)}/{int(d)}" for n, d in
                                   zip(result["count"], result["days"])], padding=4)
        ax.set(title=label, xlabel="Share of compared days (%)", xlim=(0, 116),
               xticks=[0, 25, 50, 75, 100])
    axes[0].invert_yaxis()
    fig.suptitle(f"{measure_title}: {comparison_title}")
    plt.show()


表の `Aug 1–15` と `Aug 16–31` で，分母が15日・16日になったことを確認します。両期間とも同じ順に熊谷，福岡を表示しています。

一つの日付について，選んだ方法ならどちらの地点に一日を足すかを元の値から確認します。下は8月7日です。コードの `check_date` を替えると別の日も確かめられます。


In [ ]:
check_date = "2024-08-07"
if valid_measure and valid_comparison:
    display(temperatures.loc[temperatures["date"] == check_date,
                             ["station", "date", selected_measure]])


- 確認した日付と二地点の値：
- 自分が選んだ方法では，どちらの地点の日数に入るか：

## 3. 後半も含めて説明を改める（約8分）

前半について書いた一文を，後半の結果も伝わる3〜4文へ改めます。次を含めてください。

1. 誰に何を伝えるために，どの気温・どの比較を選んだか。
2. 前半と後半の該当日数と分母。二地点の大小や差の大きさはどこまで同じだったか。
3. この月について分かったことと，別の年にも同じ傾向があると述べるために必要な観測。

（ここに記事を記入）

カーネルを再起動して上から全セルを実行し，**前半・後半の表と図が両方**再び出ることを確かめます。報告の数値をこの表へ戻して照合し，選択理由と記事も保存します。

日最低気温を選んだ場合は，一日全体の最低気温を比較しています。夜間の最低気温を意味する「熱帯夜」の日数としては報告しません。 [気象庁の気温の用語](https://www.jma.go.jp/jma/kishou/know/yougo_hp/kion.html)


<details>
<summary>選択と読み取りの手掛かり</summary>

一日の高い気温に着目するなら `selected_measure = "max_temp_c"`，最も低い気温にも着目するなら `selected_measure = "min_temp_c"` です。

例えば「35℃以上の日がどちらに多いか」という記事には，日最高気温と `comparison_kind = "threshold"` が対応します。「同じ日の日最低気温はどちらで高かったか」なら，日最低気温と `comparison_kind = "same_day"` が対応します。他の組合せも，自分の問いと合っていれば使えます。

`threshold` では両地点に該当する日があるので，二地点の日数を足しても，比べた日数にはなりません。`same_day` では一方が高い日と，同じ気温の日を合わせると，比べた日数に戻ります。前半と後半は日数が異なるため，図の棒は割合とし，ラベルには分子・分母を添えています。

</details>


## 個人の理解確認

三つの依頼を読みます。今ある値を数える，ブートストラップ，置換検定から方法を選び，繰り返し計算する場合は何を選び直し何を固定するかを一文で説明してください。この確認は20分実習の後，授業の最後に行います。

| 依頼 | データの集まり方と知りたいこと |
|---|---|
| A | 二地点とも2024年8月の全31日の気温がある。その月に日最高気温が35℃以上だった日数を報告したい。 |
| B | 同じ母集団の分布から一回ずつ独立に値を得るという調査の設定で，80件の体重を得た。この母平均を推定し，推定の幅を示したい。 |
| C | 40人のうち20人を等確率で選んで学習支援を割り当て，残りの20人と得点を比べた。誰についても支援で得点が変わらないとしたとき，観測した群の平均差以上の大きさがどれほど出るかを調べたい。 |

- A：
- B：
- C：


## 出典

気象庁「過去の気象データ検索」の日ごとの値。熊谷と福岡の2024年8月の観測値から気温列を教材用に抽出・加工したCSVを使用しました。 [熊谷の原表](https://www.data.jma.go.jp/stats/etrn/view/daily_s1.php?prec_no=43&block_no=47626&year=2024&month=8&day=&view=p1) ・ [福岡の原表](https://www.data.jma.go.jp/stats/etrn/view/daily_s1.php?prec_no=82&block_no=47807&year=2024&month=8&day=&view=p1) ・ [気象庁の利用規約](https://www.jma.go.jp/jma/kishou/info/coment.html)
